# Bucket Training Alignment Check

This notebook checks whether the bucket-prepared dataset used for training keeps `x` and `z` aligned.

It reproduces the local training path closely enough to inspect three stages:
1. the prepared bucket dataset returned by `DatasetLoader.load_dataset(...)`
2. the exact train/valid tensor datasets created by `dataset.get_dataloaders(..., return_tensor_datasets=True)`
3. the exact bucket collate step used during training

The key question is whether stored `z` still matches the actual non-pad structure in `x`.

The notebook intentionally skips CLIP/OpenCLIP cache generation for `y`, because that step does not modify the bucketed `x` or `z` tensors or the bucket-collate logic.


In [1]:
import os
import sys
from pathlib import Path
from pprint import pprint

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from IPython.display import display

from notebooks.bucket_training_alignment_helper import (
    DEFAULT_DATASET_ROOT,
    DEFAULT_TRAINING_CFG,
    build_exact_training_tensor_datasets,
    load_prepared_bucket_dataset,
    maybe_dataframe,
    summarize_manual_collate,
    summarize_prepared_bucket_dataset,
    summarize_tensor_dataset_alignment,
)


In [2]:
DATASET_ROOT = Path(DEFAULT_DATASET_ROOT)
TRAINING_CFG = Path(DEFAULT_TRAINING_CFG)
DEVICE = "cpu"
SPLIT_RATIO = 0.1
SEED = 1234
BATCH_SIZE = None
MODEL_SCALE_FACTOR = 4
MAX_MISMATCH_ROWS = 25
MAX_COLLATE_BUCKETS = 25

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATASET_ROOT =", DATASET_ROOT)
print("TRAINING_CFG =", TRAINING_CFG)

if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        f"Dataset root does not exist: {DATASET_ROOT}\n"
        "Point DATASET_ROOT to the dataset that is actually used during training."
    )


PROJECT_ROOT = /workspace/qcircuit-generation
DATASET_ROOT = /workspace/qcircuit-generation/datasets/qc_srv_dataset_3to8qubit
TRAINING_CFG = /workspace/qcircuit-generation/conf/training/paper_srv_bucket.yaml


In [3]:
loader, dataset, loader_cfg = load_prepared_bucket_dataset(
    DATASET_ROOT,
    training_cfg_path=TRAINING_CFG,
    batch_size=BATCH_SIZE,
    device=DEVICE,
)

effective_batch_size = loader_cfg["training"]["batch_size"]

print("LOADER_CFG =", loader_cfg)
print("dataset type =", type(dataset).__name__)
print("dataset.x shape =", tuple(int(dim) for dim in dataset.x.shape))
print("dataset.z shape =", tuple(int(dim) for dim in dataset.z.shape))
print("dataset.collate_fn =", getattr(dataset, "collate_fn", None))
print("dataset.bucket_batch_size =", getattr(dataset, "bucket_batch_size", None))
print("effective_batch_size =", effective_batch_size)


2026-03-24 07:42:46 - quantum_diffusion.data.dataset - INFO - Detected preprocessed dataset. Loading directly...
[INFO]: Loading tensor from `/workspace/qcircuit-generation/datasets/qc_srv_dataset_3to8qubit/dataset/ds_x.pt` onto device: cpu.
[INFO]: Loading tensor from `/workspace/qcircuit-generation/datasets/qc_srv_dataset_3to8qubit/dataset/ds_y.pt` onto device: cpu.
[INFO]: Loading tensor from `/workspace/qcircuit-generation/datasets/qc_srv_dataset_3to8qubit/dataset/ds_z.pt` onto device: cpu.
[INFO]: Instantiated config_dataset from given config on cpu.
2026-03-24 07:42:55 - quantum_diffusion.data.dataset - INFO - Dataset loaded from /workspace/qcircuit-generation/datasets/qc_srv_dataset_3to8qubit


  0%|          | 0/1 [00:00<?, ?it/s]

 - balance_tensor_dataset, njobs=1, number of samples=2463286
 - uniquify_tensor_dataset, number of samples now 2463286
 - balancing


  0%|          | 0/471 [00:00<?, ?it/s]

 - dataset size after balancing 2463286
Split: Train 9141 - Test 481 

2026-03-24 07:43:23 - quantum_diffusion.data.dataset - INFO - Datasets combined into a mixed dataset
LOADER_CFG = {'training': {'padding_mode': 'bucket', 'batch_size': 256}}
dataset type = MixedCircuitsConfigDataset
dataset.x shape = (9141, 256, 8, 52)
dataset.z shape = (9141, 256, 2)
dataset.collate_fn = cut_padding_Bucket_collate_fn
dataset.bucket_batch_size = 256
effective_batch_size = 256


In [4]:
prepared_summary = summarize_prepared_bucket_dataset(
    dataset,
    model_scale_factor=MODEL_SCALE_FACTOR,
    max_examples=MAX_MISMATCH_ROWS,
)

print("=== PREPARED DATASET ===")
pprint({k: v for k, v in prepared_summary.items() if k != "mismatch_examples"})

if prepared_summary["mismatch_examples"]:
    print("\nPREPARED DATASET MISMATCH EXAMPLES")
    display(maybe_dataframe(prepared_summary["mismatch_examples"]))
else:
    print("\nNo x/z mismatches found in the prepared bucket dataset.")


=== PREPARED DATASET ===
{'actual_nonpad_time_unique': [1,
                               2,
                               3,
                               4,
                               5,
                               6,
                               7,
                               8,
                               9,
                               10,
                               11,
                               12,
                               13,
                               14,
                               15,
                               16,
                               17,
                               18,
                               19,
                               20,
                               21,
                               22,
                               23,
                               24,
                               25,
                               26,
                               27,
                               28,
    

,flat_index,bucket_index,sample_index,z_qubits,actual_qubits,z_time,actual_nonpad_time,expected_bucket_time
0,4,0,4,4,3,4,4,4
1,6,0,6,4,4,20,5,8
2,7,0,7,4,4,20,6,8
3,9,0,9,4,4,12,5,8
4,10,0,10,4,4,20,6,8
5,17,0,17,4,4,20,11,12
6,26,0,26,4,3,8,8,8
7,28,0,28,4,4,12,8,8
8,30,0,30,4,4,16,8,8
9,31,0,31,4,3,8,8,8


In [5]:
train_ds, valid_ds = build_exact_training_tensor_datasets(
    dataset,
    split_ratio=SPLIT_RATIO,
    seed=SEED,
    batch_size=effective_batch_size,
)

train_summary = summarize_tensor_dataset_alignment(
    train_ds,
    dataset,
    split_name="train_tensor_dataset",
    model_scale_factor=MODEL_SCALE_FACTOR,
    max_examples=MAX_MISMATCH_ROWS,
)
valid_summary = summarize_tensor_dataset_alignment(
    valid_ds,
    dataset,
    split_name="valid_tensor_dataset",
    model_scale_factor=MODEL_SCALE_FACTOR,
    max_examples=MAX_MISMATCH_ROWS,
)

for name, summary in [("TRAIN TENSOR DATASET", train_summary), ("VALID TENSOR DATASET", valid_summary)]:
    print(f"\n=== {name} ===")
    pprint({k: v for k, v in summary.items() if k != "mismatch_examples"})
    if summary["mismatch_examples"]:
        print("\nMismatch examples")
        display(maybe_dataframe(summary["mismatch_examples"]))
    else:
        print("\nNo x/z mismatches found.")


[INFO]: Not balancing dataset!  balance_max=None

=== TRAIN TENSOR DATASET ===
{'actual_nonpad_time_unique': [1,
                               2,
                               3,
                               4,
                               5,
                               6,
                               7,
                               8,
                               9,
                               10,
                               11,
                               12,
                               13,
                               14,
                               15,
                               16,
                               17,
                               18,
                               19,
                               20,
                               21,
                               22,
                               23,
                               24,
                               25,
                               26,
                    

,flat_index,bucket_index,sample_index,z_qubits,actual_qubits,z_time,actual_nonpad_time,expected_bucket_time
0,7,0,7,4,4,16,6,8
1,15,0,15,4,4,20,9,12
2,19,0,19,4,4,12,8,8
3,20,0,20,4,3,8,8,8
4,26,0,26,4,4,20,12,12
5,34,0,34,4,3,12,11,12
6,39,0,39,4,4,16,12,12
7,48,0,48,4,4,20,11,12
8,49,0,49,4,4,16,5,8
9,51,0,51,4,4,8,4,4



=== VALID TENSOR DATASET ===
{'actual_nonpad_time_unique': [1,
                               2,
                               3,
                               4,
                               5,
                               6,
                               7,
                               8,
                               9,
                               10,
                               11,
                               12,
                               13,
                               14,
                               15,
                               16,
                               17,
                               18,
                               19,
                               20,
                               21,
                               22,
                               23,
                               24,
                               25,
                               26,
                               27,
                               28,

,flat_index,bucket_index,sample_index,z_qubits,actual_qubits,z_time,actual_nonpad_time,expected_bucket_time
0,0,0,0,8,8,44,28,28
1,1,0,1,8,5,36,19,20
2,2,0,2,8,8,40,36,36
3,3,0,3,8,6,8,6,8
4,4,0,4,8,6,36,11,12
5,5,0,5,8,8,28,11,12
6,6,0,6,8,5,24,14,16
7,7,0,7,8,5,36,17,20
8,8,0,8,8,7,32,22,24
9,9,0,9,8,5,12,9,12


In [6]:
train_collate = summarize_manual_collate(
    train_ds,
    dataset,
    split_name="train_collate",
    model_scale_factor=MODEL_SCALE_FACTOR,
    max_buckets=MAX_COLLATE_BUCKETS,
)
valid_collate = summarize_manual_collate(
    valid_ds,
    dataset,
    split_name="valid_collate",
    model_scale_factor=MODEL_SCALE_FACTOR,
    max_buckets=MAX_COLLATE_BUCKETS,
)

for name, summary in [("TRAIN COLLATE", train_collate), ("VALID COLLATE", valid_collate)]:
    print(f"\n=== {name} ===")
    pprint({k: v for k, v in summary.items() if k not in {"rows", "mismatch_rows"}})
    if summary["mismatch_rows"]:
        print("\nCollate mismatch rows")
        display(maybe_dataframe(summary["mismatch_rows"]))
    else:
        print("\nAll checked buckets cut to the expected shape.")



=== TRAIN COLLATE ===
{'checked_bucket_count': 25,
 'label': 'train_collate',
 'mismatch_bucket_count': 12}

Collate mismatch rows


,bucket_index,cut_shape,cut_qubits,cut_time,z_max_qubits,actual_max_qubits,z_max_time,actual_max_time,qubits_match_z,qubits_match_actual,time_match_z,time_match_actual
0,1,"(256, 8, 48)",8,48,8,8,48,44,True,True,True,False
1,2,"(256, 8, 52)",8,52,8,8,52,40,True,True,True,False
2,7,"(256, 7, 48)",7,48,7,7,48,40,True,True,True,False
3,8,"(256, 6, 40)",6,40,6,6,40,36,True,True,True,False
4,9,"(256, 8, 52)",8,52,8,8,52,40,True,True,True,False
5,11,"(256, 6, 40)",6,40,6,6,40,36,True,True,True,False
6,12,"(256, 7, 48)",7,48,7,7,48,44,True,True,True,False
7,13,"(256, 8, 48)",8,48,8,8,48,40,True,True,True,False
8,16,"(256, 6, 40)",6,40,6,6,40,36,True,True,True,False
9,20,"(256, 7, 48)",7,48,7,7,48,40,True,True,True,False



=== VALID COLLATE ===
{'checked_bucket_count': 25,
 'label': 'valid_collate',
 'mismatch_bucket_count': 13}

Collate mismatch rows


,bucket_index,cut_shape,cut_qubits,cut_time,z_max_qubits,actual_max_qubits,z_max_time,actual_max_time,qubits_match_z,qubits_match_actual,time_match_z,time_match_actual
0,0,"(256, 8, 48)",8,48,8,8,48,44,True,True,True,False
1,2,"(256, 6, 40)",6,40,6,6,40,36,True,True,True,False
2,3,"(256, 6, 40)",6,40,6,6,40,32,True,True,True,False
3,5,"(256, 8, 48)",8,48,8,8,48,44,True,True,True,False
4,6,"(256, 7, 48)",7,48,7,7,48,44,True,True,True,False
5,7,"(256, 6, 40)",6,40,6,6,40,36,True,True,True,False
6,8,"(256, 8, 52)",8,52,8,8,52,40,True,True,True,False
7,10,"(256, 8, 52)",8,52,8,8,52,40,True,True,True,False
8,11,"(256, 8, 52)",8,52,8,8,52,40,True,True,True,False
9,13,"(256, 7, 48)",7,48,7,7,48,44,True,True,True,False


In [7]:
summary_counts = {
    "prepared_any_mismatch_count": prepared_summary["any_mismatch_count"],
    "train_tensor_any_mismatch_count": train_summary["any_mismatch_count"],
    "valid_tensor_any_mismatch_count": valid_summary["any_mismatch_count"],
    "train_collate_mismatch_bucket_count": train_collate["mismatch_bucket_count"],
    "valid_collate_mismatch_bucket_count": valid_collate["mismatch_bucket_count"],
}

print("SUMMARY COUNTS")
pprint(summary_counts)

if any(value > 0 for value in summary_counts.values()):
    print(
        "\nResult: this notebook found at least one x/z or cut-shape mismatch in the bucket-training path."
    )
else:
    print(
        "\nResult: this notebook did not find x/z or cut-shape mismatches in the inspected bucket-training path."
    )


SUMMARY COUNTS
{'prepared_any_mismatch_count': 1363045,
 'train_collate_mismatch_bucket_count': 12,
 'train_tensor_any_mismatch_count': 1226479,
 'valid_collate_mismatch_bucket_count': 13,
 'valid_tensor_any_mismatch_count': 136566}

Result: this notebook found at least one x/z or cut-shape mismatch in the bucket-training path.
